In [60]:
import pandas as pd
# import matplotlib
# import matplotlib.pyplot as plt
import numpy as np
import time
import requests

def print_full(x):
    with pd.option_context('display.max_columns', None):
        display(x)
def print_all_rows(x):
    with pd.option_context('display.max_rows', None):
        display(x)
 

## Historical Price Data
* Retrieve from EOD https://eodhistoricaldata.com/
* Not worth doing without paying $80 for a month - then grab it all
* Stick with AV for now, until the other layers have been built - then come back here

* This file should be ready to go - add an API key and start loading... fundamental first then prices


TODO: use the 'from' parameter in the prices call to limit to the range of dates for which fundamental data is available

In [4]:
# EOD API key: 62c0a727bccf89.94027295
#
# Docuemtnation here:
#https://eodhistoricaldata.com/financial-apis/
apikey = '62c0a727bccf89.94027295'


# Functions related to gathering historical price data for lists of tickers
# This will be stored locally and added to on a (fairly) regular basis
# Will make a useful test dataset for prototyping methods for identifying companies to watch
def load_tickerlist():
    data = pd.read_csv('US_LIST_OF_SYMBOLS.csv')
    return data[ data['Exchange'].isin(['NASDAQ','NYSE'])]['Code'].to_list()

# load_tickers_done
# Used to generate the list of tickers already retrieved
# At some point will need to consider merging new data with existing - but for now just fill up one time
def load_fundamentals_done():
    m = pd.read_csv('eod/eod-general.csv', usecols=['ticker'])
    return m['ticker'].to_list()

# Set up an unknown function which will be written in the event that this ticker is unknown
def unknown(t):
    pd.DataFrame([t], columns=['ticker']).to_csv('eod-general.csv', mode='a', header=False)
    return 'unknown'

def load_prices(t):
    t = str.upper(t)
    
    # For testing use the testing ticker and api key
    t = 'MCD'
    apikey = 'demo'
    
    # NOTE: currently grabbing weekly data since I don't need the fine-grain of daily - can change later if needed
    url = f'https://eodhistoricaldata.com/api/eod/{t}.US?api_token={apikey}&period=w'
    print("URL:", url)
    
    # When off the free tier you can request more than one year using &from=1985-09-30
    # Grab the earliest date from the balance sheet data and use that as the from parameter
    
    try:
        csv = pd.read_csv(url, parse_dates=['Date'])
        
        # Need to test out the error handling here - what happens if/when errors occur?
        # For now just generic try/catch and assume otherwise all works ok
        
        # Remove rows that have no valid adjusted_close price - since that's the most useful number
        csv = csv[ csv['Adjusted_close'].notna() ]

        # Radical, but just store adjusted close and volume
        # I don't intend to do technical analysis on daily prices - its just the closing price the rest should be fundamentals
        # If I decide to add this later can always grab it all in one month and add it as a totally separate module
        csv = csv[['Date', 'Adjusted_close','Volume']]
        
    except:
        print('Unknown error in get call')
        return unknown(t)
    
    try:
        csv['ticker'] = t
        return csv
    
    except:
        # two possibilities here - an empty dataset (ok, just empty) or a timeout (return all null, system will check again later)
        return 'timeout'

# data_to_frames
# Accepts a EOD object containing all the fundamental data, overview, earnings and turns into dataframes
# Summary:
# overview = General
# stats = Hightlights + Valuation + SharesStats['SharesOutstanding', 'SharesFloat', 'PercentInsiders', 'PercentInstitutions']
# + Technicals['SharesShort', 'SharesShortPriorMonth', 'ShortRatio', 'ShortPercent'] + SplitsDividends['ForwardAnnualDividendRate', 'ForwardAnnualDividendYield', 'PayoutRatio', 'DividendDate', 'ExDividendDate']
# + AnalystRatings
# earnings = Earnings['reportDate', 'date', 'currency', 'epsActual', 'epsEstimate', 'epsDifference', 'surprisePercent']
# balance_sheet = 
# income_statement = 
# cash_flow = 
# df.index = pd.to_datetime(df.index)
# df.index.rename('quarter', inplace=True)
# df = df[ df.index >= '1990']
# # Convert numeric columns to integer. Doesn't work perfectly but does ok
# #        df = df.apply(pd.to_numeric, errors='ignore', downcast='integer')

def data_to_frames(data):
    
    def select_fields(src, keys):
        return { key:src[key] for key in src.keys() & keys}

    def merge_dicts(src, new):
        return (src.update(new))

    general_fields = [ 'Code', 'Name', 'Exchange', 'CurrencyCode', 'CountryName', 'CountryISO', 'FiscalYearEnd', 'IPODate', 'Sector', 'Industry', 'GicSector','GicGroup','GicIndustry','GicSubIndustry','WebURL','IsDelisted']
    overview_dict = select_fields(data['General'], general_fields)
    ov = pd.DataFrame.from_dict(overview_dict, orient='index').T.rename(columns={'Code':'ticker'}).set_index('ticker')

    shares_fields = ['SharesOutstanding', 'SharesFloat', 'PercentInsiders', 'PercentInstitutions']
    technical_fields = ['SharesShort', 'SharesShortPriorMonth', 'ShortRatio', 'ShortPercent']
    splits_fields = ['ForwardAnnualDividendRate', 'ForwardAnnualDividendYield', 'PayoutRatio', 'DividendDate', 'ExDividendDate']
    stats_dict = data['Highlights']
    merge_dicts(stats_dict, data['Valuation'])
    merge_dicts(stats_dict, select_fields(data['SharesStats'], shares_fields))
    merge_dicts(stats_dict, select_fields(data['Technicals'], technical_fields))
    merge_dicts(stats_dict, select_fields(data['SplitsDividends'], splits_fields))
    merge_dicts(stats_dict, data['AnalystRatings'])
    stats = pd.DataFrame.from_dict(stats_dict, orient='index').T.rename(columns={'MostRecentQuarter':'Quarter'}).set_index('Quarter')

    earnings_dict = data['Earnings']['History']
    earnings = pd.DataFrame.from_dict(data['Earnings']['History'], orient='index').drop(columns=['date']).dropna(axis='columns', how='all')
    earnings = earnings[ earnings['epsActual'].notna() ]
    earnings.rename_axis(index='Quarter')

    bs_dict = data['Financials']['Balance_Sheet']['quarterly']
    bs = pd.DataFrame.from_dict(bs_dict, orient='index').drop(columns=['date']).dropna(axis='columns', how='all')
    bs.rename_axis(index='Quarter')
    inc_dict = data['Financials']['Income_Statement']['quarterly']
    inc = pd.DataFrame.from_dict(inc_dict, orient='index').drop(columns=['date']).dropna(axis='columns', how='all')
    inc.rename_axis(index='Quarter')
    cf_dict = data['Financials']['Cash_Flow']['quarterly']
    cf = pd.DataFrame.from_dict(cf_dict, orient='index').drop(columns=['date']).dropna(axis='columns', how='all')
    cf.rename_axis(index='Quarter')

    return [ov, stats, earnings, bs, inc, cf]

        
# Re-use the earnings function for the other fundamental data
# URL: function=BALANCE_SHEET
# Keys: annualEarnings -> annualReports
def load_fundamentals(t):
    t = str.upper(t)

    # For testing just use the demo URL and ticker AAPL
#     t = 'AAPL'
#     apikey = 'demo'

    # For now assume it will always be a .US exchange so hard-code, but maybe in future this will need to be added...
    url = f'https://eodhistoricaldata.com/api/fundamentals/{t}.US?api_token={apikey}'
    print("URL:", url)

    
    # In case of any errors fetching the data - just return unknown so we ignore this item
    try:
        r = requests.get(url)
        
        # r.status_code of 402 indicates an API error - permissions issue (NOT unknown)
        # In this case maybe time to stop running and figure out why error...
        # 
        if (r.status_code == 402):
            return '402 error'
        data = r.json()

    except:
        print('Unknown error in get call')
        return unknown(t)
    
    print(data.keys())

    # Code up the various errors later once I know how the EOD API works...
    # Timeout returns an Information text
    if ('Information' in data.keys()):
        return 'timeout'

    # Other errors are probably due to some issue with this ticker - drop it
    if ('Error Message' in data.keys()):
        return unknown(t)
    
    if ('General' not in data.keys()):
        return unknown(t)
    
    # We have some data - fornwo assume that if we have 'General' then we have all the others
    [ov, stats, earnings, bs, inc, cf] = data_to_frames(data)
    
    # Add the ticker symbol to frames (ov not needed as it already has the ticker)
    stats['ticker'] = t
    earnings['ticker'] = t
    bs['ticker'] = t
    inc['ticker'] = t
    cf['ticker'] = t
    
    # Slightly different tactic here - directly save the data (append to files) and return a string value
    ov.to_csv('eod/eod-general.csv', mode='a', header=False)
    stats.to_csv('eod/eod-stats.csv', mode='a', header=False)
    earnings.to_csv('eod/eod-earnings.csv', mode='a', header=False)
    bs.to_csv('eod/eod-balancesheet.csv', mode='a', header=False)
    inc.to_csv('eod/eod-incomestatement.csv', mode='a', header=False)
    cf.to_csv('eod/eod-cashflow.csv', mode='a', header=False)

    return t

def load_next_ticker(fulllist=np.nan, tickers_so_far=np.nan):
    if (fulllist is np.nan):
        fulllist = load_tickerlist()
    if (tickers_so_far is np.nan):
        tickers_so_far = pd.read_csv('EOD-prices.csv')['ticker'].unique()
    main_list = np.setdiff1d(fulllist, tickers_so_far)
    print('Tickers remaining:', len(main_list))
    if (len(main_list) == 0):
        return 'DONE'
    return load_prices(main_list[0])

def load_next_fundamentals(alltickers, tickers_done):
    try:
        item = alltickers.pop(0)
        while (item in tickers_done):
            item = alltickers.pop(0)
    except:
        print('*** DONE! ***')
        return "DONE"
    print('Full list:', len(alltickers), '  Tickers so far:', len(tickers_done))
    return load_fundamentals(item)

    
# add_batch_prices: performs ALL the work required to grab some more data
# including reading all the relevant files and saving again...
def add_batch_prices(r=100, r2=5):
    master = load_master()
    alltickers = load_tickerlist()
    for i in range(0,r):
        for j in range(0,r2):
            next_ticker = load_next_ticker(alltickers, master['ticker'].unique() )
            if (type(next_ticker) is str):
                break
            master = pd.concat([master,next_ticker])
        # if not last time through loop then sleep (prevents unneccesary minute wait at the end)
        if (i+1 < r):
            time.sleep(60)
    # moved this line to the end to save poor hard drive from continually re-writing large amounts of data
    # only problem could be if any errors occur before this point the file won't be saved, but it seems quite robust now...
    save_master(master)

# Load the tickerlist_general rather than tickerlist_earnings - uses the master tickerlist to remove unknowns
def add_batch_fundamentals(r=3):
    tickers_done = load_fundamentals_done()
    alltickers = load_tickerlist()
    for i in range(0,r):
        ret = load_next_fundamentals(alltickers, tickers_done)
        print('Ret:', ret)
        if (ret == 'unknown'):
            print('Unknown ticker - stored a blank record')
        elif (ret == 'timeout'):
            print('Timeout - this shouldnt happen wtih EOD...')
        elif (ret == '402 error'):
            print('402: Something wrong, stopping...')
            break;
        else:
            print('Success - ', ret, ' added to tickers_done')
            tickers_done.append(ret)
    print('Batch DONE')

    
# Run it...
#add_batch_fundamentals()

# Testing the prices module
aapl = load_prices('AAPL')
aapl

URL: https://eodhistoricaldata.com/api/eod/MCD.US?api_token=demo&period=w


,Date,Adjusted_close,Volume,ticker
0,1966-07-05,0.1434,4179600,MCD
1,1966-07-11,0.1365,1956150,MCD
2,1966-07-18,0.1296,1786050,MCD
3,1966-07-25,0.1259,1251450,MCD
4,1966-08-01,0.1269,753300,MCD
...,...,...,...,...
2988,2023-10-09,248.3100,15006100,MCD
2989,2023-10-16,258.1100,19385800,MCD
2990,2023-10-23,255.7600,14548500,MCD
2991,2023-10-30,267.8700,21074600,MCD


In [29]:
# EOD List of tickers
url = f'https://eodhd.com/api/exchange-symbol-list/US?api_token={apikey}&delisted=1&fmt=csv'
print("URL:", url)
    
# Grabbed a copy and stored in eod-symbol_list.csv
csv = pd.read_csv('EOD/eod-symbol-list.csv')

csv

URL: https://eodhd.com/api/exchange-symbol-list/US?api_token=62c0a727bccf89.94027295&delisted=1&fmt=csv


,Code,Name,Country,Exchange,Currency,Type,Isin
0,-P-S,-P-S,USA,NYSE,USD,Preferred Stock,NaN
1,!DJI,TRS DJ Industrial Average PR USD,USA,NaN,USD,Common Stock,NaN
2,00779G392,00779G392,USA,NMFQS,USD,Mutual Fund,NaN
3,0NPP,Koninklijke DSM NV,USA,NaN,USD,Common Stock,NaN
4,1836930D,AMALGAMATED BANK,USA,NaN,USD,Common Stock,NaN
...,...,...,...,...,...,...,...
58840,ZZLLD,ZZLL Information Technology Inc,USA,OTCBB,USD,Common Stock,NaN
58841,ZZPWF,ZZPWF,USA,OTCGREY,USD,Common Stock,NaN
58842,ZZZ,TEST TICKER FOR UTP,USA,NYSE ARCA,USD,Common Stock,NaN
58843,ZZZOD,Zinc One Resources Inc,USA,PINK,USD,Common Stock,NaN


In [32]:
# Question: what's the difference between what I just stored (eod-symbol-list.csv) and what was stored ages ago in US_LIST_OF_SYMBOLS.csv) ?
sl = pd.read_csv('EOD/US_LIST_OF_SYMBOLS.csv')
sl

,Code,Name,Country,Exchange,Currency,Type,Isin
0,A,Agilent Technologies Inc,USA,NYSE,USD,Common Stock,US00846U1016
1,AA,Alcoa Corp,USA,NYSE,USD,Common Stock,US0138721065
2,AAA,Listed Funds Trust - AAF First Priority CLO Bo...,USA,NYSE ARCA,USD,ETF,US53656F6566
3,AAAAX,DEUTSCHE REAL ASSETS FUND CLASS A,USA,NMFQS,USD,FUND,NaN
4,AAACX,AAACX,USA,NASDAQ,USD,FUND,NaN
...,...,...,...,...,...,...,...
52757,ZZHGF,ZhongAn Online P & C Insurance Co. Ltd,USA,PINK,USD,Common Stock,NaN
52758,ZZHGY,ZhongAn Online P & C Insurance Co. Ltd,USA,PINK,USD,Common Stock,NaN
52759,ZZLL,ZZLL Information Technology Inc,USA,OTCQB,USD,Common Stock,US98880P2020
52760,ZZZOF,Zinc One Resources Inc,USA,PINK,USD,Common Stock,NaN


In [42]:
slcodes = set(sl['Code'].to_list())
slcodes

csvcodes = set(csv['Code'].to_list())
csvcodes

slcodes

{'DODRW',
 'PTMEY',
 'TNGRF',
 'IGIIX',
 'APIUX',
 'MAFOX',
 'AICYX',
 'PDSKX',
 'CULRX',
 'PXTIX',
 'PBCO',
 'BSX-PA',
 'NABIX',
 'FIIUX',
 'RMAX',
 'TRS',
 'HMXAX',
 'AGMH',
 'RKGXF',
 'SAFT',
 'DBEU',
 'WAIOX',
 'WMSCX',
 'WISIX',
 'CECPX',
 'CTTLX',
 'CBDX',
 'IGOV',
 'NXHSF',
 'PRAY',
 'KEYR',
 'SMID',
 'RELY',
 'CIZ',
 'INFRX',
 'CAGXX',
 'MRGCX',
 'UEXCF',
 'IAMOX',
 'TVIXF',
 'PSA-P-H',
 'CTTAF',
 'CUIRF',
 'NSRIX',
 'DVFTX',
 'PLTJX',
 'FTGLX',
 'LGRXX',
 'MLVAX',
 'MRSRX',
 'FIKCX',
 'HARXX',
 'HLDTX',
 'WDH',
 'RMYHY',
 'MYE',
 'CHSH',
 'ASXDX',
 'PARHX',
 'FIONX',
 'LIHKX',
 'ACIOX',
 'RRBGX',
 'VWLTX',
 'VSTSX',
 'TCBRX',
 'HDB',
 'PSNY',
 'CGOCX',
 'EITMX',
 'AGVYX',
 'UMCVX',
 'WELX',
 'ALORY',
 'BAK',
 'SCPPF',
 'JLJHX',
 'HAGHY',
 'BTCMX',
 'FMAO',
 'ALCCX',
 'DCF',
 'FMCEX',
 'UNRV',
 'GRPTF',
 'RYALX',
 'TIRRX',
 'XBJL',
 'PTEL',
 'OIERX',
 'JCRAX',
 'HADRX',
 'THOGF',
 'BNOV',
 'OSCUF',
 'SHW',
 'FCLKX',
 'CMOMX',
 'GTRCX',
 'MDBRX',
 'ACOGF',
 'SNTFX',
 'TNMRX',
 '

In [56]:
# Well, that's weird...
# Seems that these two files contain almost completely different codes ???
for code in sl['Code'][0:10]:
    print(code)
    print(csv[ csv['Code'] == code])

A
Empty DataFrame
Columns: [Code, Name, Country, Exchange, Currency, Type, Isin]
Index: []
AA
Empty DataFrame
Columns: [Code, Name, Country, Exchange, Currency, Type, Isin]
Index: []
AAA
Empty DataFrame
Columns: [Code, Name, Country, Exchange, Currency, Type, Isin]
Index: []
AAAAX
Empty DataFrame
Columns: [Code, Name, Country, Exchange, Currency, Type, Isin]
Index: []
AAACX
Empty DataFrame
Columns: [Code, Name, Country, Exchange, Currency, Type, Isin]
Index: []
AAAEX
Empty DataFrame
Columns: [Code, Name, Country, Exchange, Currency, Type, Isin]
Index: []
AAAFX
Empty DataFrame
Columns: [Code, Name, Country, Exchange, Currency, Type, Isin]
Index: []
AAAGX
Empty DataFrame
Columns: [Code, Name, Country, Exchange, Currency, Type, Isin]
Index: []
AAAHX
Empty DataFrame
Columns: [Code, Name, Country, Exchange, Currency, Type, Isin]
Index: []
AAAIX
Empty DataFrame
Columns: [Code, Name, Country, Exchange, Currency, Type, Isin]
Index: []


In [61]:
print_all_rows(csv.head(n=200))

,Code,Name,Country,Exchange,Currency,Type,Isin
0,-P-S,-P-S,USA,NYSE,USD,Preferred Stock,NaN
1,!DJI,TRS DJ Industrial Average PR USD,USA,NaN,USD,Common Stock,NaN
2,00779G392,00779G392,USA,NMFQS,USD,Mutual Fund,NaN
3,0NPP,Koninklijke DSM NV,USA,NaN,USD,Common Stock,NaN
4,1836930D,AMALGAMATED BANK,USA,NaN,USD,Common Stock,NaN
5,1885708D,MEREDITH HOLDINGS CORP,USA,NaN,USD,Common Stock,NaN
6,1922527D,TRINSEO PLC,USA,NaN,USD,Common Stock,NaN
7,1MSF,1MSF,USA,NaN,USD,Common Stock,NaN
8,202K1,S K1 Cbot Soybean Futures May 2021,USA,NaN,USD,Common Stock,NaN
9,2104381D,SINTX TECHNOLOGIES INC,USA,NaN,USD,Common Stock,NaN


In [67]:
sl[ sl['Name'].str.contains('pple')]

,Code,Name,Country,Exchange,Currency,Type,Isin
195,AAPL,Apple Inc,USA,NASDAQ,USD,Common Stock,US0378331005
1308,AGPL,Apple Green Holding Inc,USA,PINK,USD,Common Stock,NaN
2525,APLE,Apple Hospitality REIT Inc,USA,NYSE,USD,Common Stock,US03784Y2000
2585,APRU,Apple Rush Company,USA,PINK,USD,Common Stock,US03785R2040
18444,GAPJ,Golden Apple Oil & Gas Inc,USA,PINK,USD,Common Stock,NaN
19378,GJS,STRATSSM Certificates series supplement 2006-2...,USA,NYSE,USD,Common Stock,US86311R3012
30888,MLP,Maui Land & Pineapple Company Inc,USA,NYSE,USD,Common Stock,US5773451019
36383,PEGY,Pineapple Holdings Inc,USA,NASDAQ,USD,Common Stock,US72303P1075
37746,PNPL,Pineapple Express,USA,OTCCE,USD,Common Stock,US72302T1007


In [70]:
csv.dropna(subset='Name', inplace=True)
csv[ csv['Name'].str.contains('pple')]

,Code,Name,Country,Exchange,Currency,Type,Isin
1482,AGPL,Apple Green Holding Inc,USA,PINK,USD,Common Stock,NaN
2825,AOIA,Apple Orthodontix Inc,USA,NYSE,USD,Common Stock,NaN
3051,APWD,Applewoods Inc,USA,NASDAQ,USD,Common Stock,NaN
13710,DPS,Dr Pepper Snapple Group Inc,USA,NYSE,USD,Common Stock,US26138E1091
50965,TAVI,Thorn Apple Valley Inc,USA,NASDAQ,USD,Common Stock,NaN
